In [1]:
# Parameters
BATCH_MODE = True


# LTX-2 - Generation Audiovisuelle Conjointe (Video + Audio)

**Module :** 02-Video-Advanced
**Niveau :** Avance
**Technologies :** LTX-2 (Lightricks), ltx-core + ltx-pipelines, Gemma 3, torch, bitsandbytes
**Duree estimee :** 60 minutes
**VRAM :** ~16-24 GB (INT8/NF4 obligatoire, le modele 22B ne tient pas en FP16 sur 24 GB)

## Objectifs d'Apprentissage

- [ ] Comprendre l'architecture LTX-2 et la generation audiovisuelle conjointe
- [ ] Installer les packages Lightricks ltx-core + ltx-pipelines (non standards vs diffusers)
- [ ] Charger le checkpoint 22B avec quantization INT8/NF4 pour tenir en VRAM
- [ ] Generer une video avec bande-son synchronisee en une seule passe de diffusion
- [ ] Comparer l'approche jointe LTX-2 vs le pipeline traditionnel (video-gen puis TTS/foley separe)
- [ ] Estimer la VRAM et planifier une configuration realisable

## Prerequis

- GPU avec 20+ GB VRAM (RTX 3090 24GB minimum avec quantization INT4/NF4)
- Notebook 02-2 (LTX-Video 0.9) complete pour comparaison
- Checkpoint LTX-2.3 telecharge (~44 GB) + spatial upsampler
- **Gemma 3 12B telecharge** (text encoder, necessite acceptation licence sur HuggingFace)
- Packages : `torch`, `bitsandbytes`, `imageio`, `imageio-ffmpeg`, `av`

**Navigation :** [<< 02-4 SVD](02-4-SVD-Image-to-Video.ipynb) | [Index](../README.md) | [Comparaison >>](../03-Orchestration/03-1-Multi-Model-Video-Comparison.ipynb)


In [2]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"

# Modele LTX-2.3 (Lightricks, HF: Lightricks/LTX-2.3)
model_repo = "Lightricks/LTX-2.3"
model_file = "ltx-2.3-22b-dev.safetensors"        # Checkpoint principal 22B
spatial_upscaler_file = "ltx-2.3-spatial-upscaler-x1.5-1.0.safetensors"
gemma_repo = "google/gemma-3-12b-it-qat-q4_0-unquantized"  # Text encoder (gated HF)
device = "cuda"

# Quantization : le 22B (~44 GB FP16) ne tient pas sur 24 GB => INT4/NF4 obligatoire
quantization = "int4_nf4"          # "int4_nf4", "int8", "fp16" (fp16 = 44 GB, GPU 48GB+ requis)

# Parametres generation
num_frames = 97                    # LTX-2 supporte des sequences longues (~4s a 24 fps)
num_inference_steps = 30           # Etapes de debruitage
guidance_scale = 3.0               # CFG scale (LTX-2 prefere un CFG bas)
height = 512                       # Hauteur video
width = 768                        # Largeur video
fps_output = 24                    # FPS de sortie
audio = True                       # Generation audio conjoint (feature cle de LTX-2)

# Configuration
run_generation = True              # Executer la generation (False si modele/Gemma manquant)
save_as_mp4 = True                 # Sauvegarder en MP4 avec audio embarque
save_results = True

# Print informatif de confirmation (convention outputs systematiques)
print(f"LTX-2 Audiovisual | quantization={quantization} | {num_frames}f @ {width}x{height} | audio={audio}")


LTX-2 Audiovisual | quantization=int4_nf4 | 97f @ 768x512 | audio=True


In [3]:
# Parameters
BATCH_MODE = True


In [4]:
# Setup environnement et imports
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import logging

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Chargement robuste de la configuration .env (recherche recursive des parents,
# necessaire car Papermill change le cwd).
from dotenv import load_dotenv
current_path = Path.cwd()
GENAI_ROOT = current_path
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path}")
        env_loaded = True
        GENAI_ROOT = current_path
        break
    current_path = current_path.parent
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")
    GENAI_ROOT = GENAI_ROOT.parent

# Helpers GenAI (pattern canonique, cf 02-2).
HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.genai_helpers import setup_genai_logging
        print("Helpers GenAI importes")
    except ImportError:
        print("Helpers GenAI non disponibles - mode autonome")

OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'ltx2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('ltx2')

print(f"LTX-2 - Generation Audiovisuelle Conjointe")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}")
print(f"Modele : {model_repo} ({model_file})")
print(f"Quantization : {quantization} | Frames : {num_frames}, Steps : {num_inference_steps}, CFG : {guidance_scale}")


LTX-2 - Generation Audiovisuelle Conjointe
Date : 2026-06-15 16:08:53
Mode : interactive
Modele : Lightricks/LTX-2.3 (ltx-2.3-22b-dev.safetensors)
Quantization : int4_nf4 | Frames : 97, Steps : 30, CFG : 3.0


## Licence : LTX-2 Community License (important)

LTX-2.3 est distribue sous la **LTX-2 Community License Agreement** (date de licence : 5 janvier 2026).
Ce n'est **PAS** une licence OSI approuvee (contrairement a MIT/Apache). Points cles a comprendre :

| Aspect | Detail |
|--------|--------|
| **Type** | Licence communautaire custom Lightricks (non-OSI) |
| **Usage recherche/education** | Autorise (cas de cette serie pedagogique) |
| **Seuil commercial** | Au-dela d'un seuil de revenu, une licence commerciale separée de Lightricks est requise |
| **Derives** | Les poids fine-tunes / modeles distilles restent soumis a cet accord |
| **Donnees d'entrainement** | NON couvertes par cet accord ( Data) |

> **Avant tout deploiement commercial** : lire l'integralite du fichier LICENSE sur [HuggingFace LTX-2.3](https://huggingface.co/Lightricks/LTX-2.3) et [GitHub Lightricks/LTX-2](https://github.com/Lightricks/LTX-2). Au-dela du seuil de revenu, contactez Lightricks pour une licence commerciale.

Cette serie l'utilise dans un cadre strictement pedagogique (generation d'exemples synthetiques neutres), conforme a l'usage autorise.


In [5]:
# Chargement .env et verification GPU + dependances LTX-2
print("\n--- VERIFICATION GPU ---")
print("=" * 40)

# L'init CUDA peut declencher une compilation JIT triton (backend CUDA de torch).
# Sur un env sans compilateur C (gcc/clang), on capture le diagnostic au lieu de crasher.
cuda_ok = False
cuda_diag = ""
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
        vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3
        print(f"GPU : {gpu_name}")
        print(f"VRAM totale : {vram_total:.1f} GB")
        print(f"VRAM libre : {vram_free:.1f} GB")
        print(f"CUDA : {torch.version.cuda}")
        if vram_total < 20:
            print(f"\nATTENTION : VRAM ({vram_total:.0f} GB) < 20 GB. La quantization INT4/NF4 est OBLIGATOIRE")
            print("  pour faire tenir le modele 22B (~44 GB en FP16). Sans elle, OOM garanti.")
    else:
        print("CUDA non disponible (torch.cuda.is_available()=False).")
        cuda_diag = "torch.cuda.is_available()=False"
except Exception as e:
    cuda_diag = f"{type(e).__name__}: {str(e)[:200]}"
    print(f"Init CUDA bloque : {cuda_diag}")
    if "C compiler" in str(e) or "triton" in str(e).lower():
        print("  Cause : le backend CUDA triton de torch veut JIT-compiler un kernel et ne trouve")
        print("  pas de compilateur C (gcc/clang) dans le PATH. Installer m2w64-toolchain ou clang,")
        print("  ou executer sur un env Linux avec toolchain CUDA complete.")

if not cuda_ok:
    print("\nLTX-2 necessite un GPU operationnel (20+ GB VRAM avec quantization). Mode pedagogique.")
    run_generation = False
    device = "cpu"

print("\n--- VERIFICATION DEPENDANCES LTX-2 ---")
print("=" * 40)

deps_ok = True

# Packages LTX (API custom Lightricks, NON diffusers).
for pkg in ("ltx_core", "ltx_pipelines"):
    try:
        __import__(pkg)
        print(f"{pkg} : OK")
    except ImportError:
        print(f"{pkg} NON INSTALLE (pip install -e packages/{pkg} depuis le repo Lightricks/LTX-2)")
        deps_ok = False

for pkg_name, import_name in [("bitsandbytes", "bitsandbytes"), ("imageio", "imageio"), ("av", "av")]:
    try:
        mod = __import__(import_name)
        ver = getattr(mod, "__version__", "OK")
        print(f"{pkg_name} : {ver}")
    except ImportError:
        print(f"{pkg_name} NON INSTALLE")
        deps_ok = False
    except Exception as e:
        # bitsandbytes importe ses modules triton au top-level ; sans compilateur C
        # (gcc/clang) le JIT triton echoue. On capture le diagnostic sans crasher.
        print(f"{pkg_name} : present mais init bloque ({type(e).__name__}: {str(e)[:80]})")
        if pkg_name == "bitsandbytes":
            deps_ok = False
            print("  bitsandbytes requiert un compilateur C pour ses kernels triton INT8/NF4.")
            print("  Sans lui, la quantization LTX-2 ne peut s'appliquer => generation non executee ce cycle.")

if not deps_ok:
    print("\nDependances LTX-2 manquantes. Le notebook montrera le code et l'API sans executer.")
    run_generation = False

print(f"\nDevice : {device}")
print(f"Generation activee : {run_generation}")
print(f"Quantization : {quantization}")
print(f"Diagnostic CUDA : {cuda_diag or 'OK'}")



--- VERIFICATION GPU ---


GPU : NVIDIA GeForce RTX 3090
VRAM totale : 24.0 GB
VRAM libre : 24.0 GB
CUDA : 12.8

--- VERIFICATION DEPENDANCES LTX-2 ---
ltx_core NON INSTALLE (pip install -e packages/ltx_core depuis le repo Lightricks/LTX-2)
ltx_pipelines NON INSTALLE (pip install -e packages/ltx_pipelines depuis le repo Lightricks/LTX-2)


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\triton\windows_utils.py:433: UserWarning: Failed to find CUDA.
  warnings.warn("Failed to find CUDA.")


bitsandbytes : present mais init bloque (RuntimeError: Failed to find C compiler. Please specify via CC environment variable.)
  bitsandbytes requiert un compilateur C pour ses kernels triton INT8/NF4.
  Sans lui, la quantization LTX-2 ne peut s'appliquer => generation non executee ce cycle.
imageio : 2.37.2


av : 14.3.0

Dependances LTX-2 manquantes. Le notebook montrera le code et l'API sans executer.

Device : cuda
Generation activee : False
Quantization : int4_nf4
Diagnostic CUDA : OK


## Section 1 : Architecture LTX-2 et generation audiovisuelle conjointe

LTX-2 (Lightricks) est le **premier modele fondationnel audio-video base sur DiT** (Diffusion Transformer). La difference essentielle avec tous les autres notebooks de cette serie : il genere **video ET audio synchronises dans une seule passe de diffusion**, au lieu d'une video muette qu'il faut ensuite sonoriser separement.

| Composant | Description |
|-----------|-------------|
| **Architecture** | DiT (Diffusion Transformer) 22B parametres, audio-video conjoint |
| **Text encoder** | Gemma 3 12B (Google, encodeur du prompt) |
| **Sortie** | Video + bande-son synchronisees (un seul fichier MP4) |
| **Upscalers** | Spatial x1.5 / x2 + temporal x2 (post-processing) |
| **Package API** | ltx-core + ltx-pipelines (custom Lightricks, NON diffusers) |

### LTX-2 vs les autres modeles de la serie Video

| Aspect | LTX-2 (02-5) | LTX-Video 0.9 (02-2) | HunyuanVideo (02-1) |
|--------|--------------|----------------------|---------------------|
| **Audio** | Oui, conjoint (cle) | Non | Non |
| **VRAM** | ~16-24 GB (INT4/NF4) | ~8 GB | ~18 GB (INT8) |
| **Parametres** | 22B | ~2B | ~13B |
| **Qualite** | SOTA | Correcte | Tres bonne |
| **API** | ltx-pipelines (CLI module) | diffusers | ComfyUI / diffusers |
| **Licence** | LTX-2 Community (non-OSI) | Ouverte | Open |

La nouveaute majeure est la **colonne Audio** : aucun autre modele de la serie ne genere de son. Traditionnellement, une video generee doit etre sonorisee dans un pipeline secondaire (TTS pour la voix, foley pour les effets, musique). LTX-2 supprime cette etape en produisant les deux de maniere coherente dans la meme passe.


In [6]:
# Chargement du pipeline LTX-2 (API ltx-pipelines, NON diffusers)
#
# LTX-2 s'invoque via un module CLI (python -m ltx_pipelines.ti2vid_two_stages ...)
# On prepare les chemins vers les checkpoints + Gemma, puis on appelle le pipeline.
pipe_ltx2 = None
ltx2_ready = False

if run_generation:
    print("\n--- CHARGEMENT DU PIPELINE LTX-2 ---")
    print("=" * 40)

    # Resolution des chemins checkpoints depuis le cache HuggingFace.
    from huggingface_hub import hf_hub_download

    hf_token = os.getenv("HUGGINGFACE_TOKEN")
    try:
        print(f"Localisation checkpoint : {model_repo}/{model_file} ...")
        checkpoint_path = hf_hub_download(repo_id=model_repo, filename=model_file, token=hf_token)
        print(f"  -> {checkpoint_path}")

        print(f"Localisation spatial upsampler : {spatial_upscaler_file} ...")
        spatial_upsampler_path = hf_hub_download(repo_id=model_repo, filename=spatial_upscaler_file, token=hf_token)
        print(f"  -> {spatial_upsampler_path}")

        # Gemma 3 (text encoder, GATED sur HF => necessite acceptation licence prealable).
        print(f"Localisation Gemma text encoder : {gemma_repo} ...")
        try:
            gemma_root = hf_hub_download(repo_id=gemma_repo, filename="config.json", token=hf_token)
            gemma_root = str(Path(gemma_root).parent)
            print(f"  -> {gemma_root}")
        except Exception as e:
            print(f"  ECHEC Gemma : {type(e).__name__}: {str(e)[:150]}")
            print("  Gemma 3 est GATED sur HuggingFace. Le user doit accepter la licence sur")
            print("  https://huggingface.co/google/gemma-3-12b-it-qat-q4_0-unquantized puis")
            print("  propager l'approbation au token. Sans Gemma, pas de generation.")
            gemma_root = None

        ltx2_ready = (gemma_root is not None)

        if device == "cuda":
            vram_used = torch.cuda.memory_allocated(0) / 1024**3
            print(f"  VRAM apres resolution checkpoints : {vram_used:.1f} GB")
    except Exception as e:
        print(f"Erreur localisation checkpoints : {type(e).__name__}: {str(e)[:200]}")
        print("Verifiez que les checkpoints LTX-2.3 sont telecharges (~44 GB).")
        run_generation = False
else:
    print("Chargement pipeline desactive")
    print(f"Generation activee : {run_generation}")


Chargement pipeline desactive
Generation activee : False


## Section 2 : Generation text-to-audiovideo

LTX-2 prend un prompt textuel et produit un fichier MP4 contenant **la video ET sa bande-son synchronisees**. On invoque le pipeline deux-etages recommande (`ti2vid_two_stages`) qui produit une meilleure coherence qu'un pipeline mono-stage.

Le pipeline s'execute comme un sous-module (CLI Lightricks). On encapsule l'appel dans une fonction Python qui reconstruit la commande et capture le fichier de sortie.


In [7]:
# Generation text-to-audiovideo conjointe
def generate_ltx2_audiovideo(prompt: str, negative_prompt: str = "",
                              seed: int = 42, gen_audio: bool = True) -> Dict[str, Any]:
    '''
    Genere une video + audio synchronises avec LTX-2 via le pipeline ltx-pipelines.

    Args:
        prompt: Description textuelle de la scene ET du son souhaites
        negative_prompt: Elements a eviter
        seed: Graine aleatoire pour reproductibilite
        gen_audio: Inclure la bande-son conjointe (feature cle de LTX-2)

    Returns:
        Dict avec output_path, temps de generation et metadonnees
    '''
    if not ltx2_ready:
        return {"success": False, "error": "Pipeline LTX-2 non pret (checkpoints/Gemma manquants)"}

    output_path = OUTPUT_DIR / f"ltx2_{seed}.mp4"
    cmd = [
        sys.executable, "-m", "ltx_pipelines.ti2vid_two_stages",
        "--checkpoint-path", str(checkpoint_path),
        "--spatial-upsampler-path", str(spatial_upsampler_path),
        "--gemma-root", str(gemma_root),
        "--prompt", prompt,
        "--output-path", str(output_path),
        "--num_frames", str(num_frames),
        "--num_inference_steps", str(num_inference_steps),
        "--guidance_scale", str(guidance_scale),
        "--height", str(height),
        "--width", str(width),
        "--seed", str(seed),
    ]
    if not gen_audio:
        cmd.append("--no-audio")
    if negative_prompt:
        cmd += ["--negative-prompt", negative_prompt]

    print(f"Prompt : {prompt[:80]}")
    print(f"Parametres : {num_frames} frames, {num_inference_steps} steps, CFG={guidance_scale}")
    print(f"Resolution : {width}x{height} | Audio : {gen_audio}")
    print("Generation en cours (22B, peut prendre plusieurs minutes)...")

    try:
        if device == "cuda":
            torch.cuda.reset_peak_memory_stats()
        start_time = time.time()
        import subprocess
        result_proc = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
        gen_time = time.time() - start_time

        if result_proc.returncode != 0:
            return {"success": False,
                    "error": f"pipeline exit {result_proc.returncode}: {result_proc.stderr[:300]}"}

        if not output_path.exists():
            return {"success": False, "error": "fichier de sortie non cree"}

        size_mb = output_path.stat().st_size / 1024**2
        out = {
            "success": True,
            "output_path": str(output_path),
            "generation_time": gen_time,
            "size_mb": size_mb,
            "prompt": prompt, "seed": seed, "audio": gen_audio,
            "params": {"num_frames": num_frames, "guidance_scale": guidance_scale,
                       "num_inference_steps": num_inference_steps,
                       "height": height, "width": width},
        }
        if device == "cuda":
            out["vram_peak"] = torch.cuda.max_memory_allocated(0) / 1024**3
        return out
    except Exception as e:
        return {"success": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}


# Premier test : scene avec son identifiable (vagues + mouettes).
prompt_1 = ("ocean waves crashing on a rocky shore at sunset, seagulls calling overhead, "
            "cinematic, warm golden light, natural ambient sound")

if run_generation and ltx2_ready:
    result_1 = generate_ltx2_audiovideo(prompt_1, seed=42)
    if result_1['success']:
        print(f"\nGeneration reussie")
        print(f"  Temps total : {result_1['generation_time']:.1f}s")
        print(f"  Taille : {result_1['size_mb']:.1f} MB")
        print(f"  Sortie : {result_1['output_path']}")
        if 'vram_peak' in result_1:
            print(f"  VRAM pic : {result_1['vram_peak']:.1f} GB")
    else:
        print(f"Erreur : {result_1['error']}")
else:
    print("Generation desactivee (modeles manquants ou env incomplet)")
    print(f"\nExemple d'appel :")
    print(f"  result = generate_ltx2_audiovideo('{prompt_1[:50]}...', seed=42)")


Generation desactivee (modeles manquants ou env incomplet)

Exemple d'appel :
  result = generate_ltx2_audiovideo('ocean waves crashing on a rocky shore at sunset, s...', seed=42)


## Interpretation : pourquoi la generation conjointe change tout

### Le pipeline traditionnel (video muette puis sonorisation)

Sans LTX-2, produire une video avec du son demande une **chaine multi-etapes** :

1. Generation video (modele video-only : HunyuanVideo, Wan, SVD) -> video muette
2. TTS pour les voix (notebook Audio/Kokoro, FishAudio)
3. Foley / effets sonores (banque de sons ou generation dediee)
4. Mixage et **synchronisation manuelle** (lip-sync, alignement temporal)

Chaque etape ajoute des erreurs de synchronisation. Le lip-sync est notoirement difficile : la bouche d'un personnage doit matcher exactement le son produit, sous peine d'un effet "film doube" degrade.

### L'approche jointe LTX-2

LTX-2 produit **les deux modalites dans la meme passe de diffusion**. Le DiT 22B apprend conjointement la coherence visuelle ET audio-visual (un coup de marteau = un son de choc synchrone). Le MP4 de sortie embarque directement la piste audio alignee au frame pres.

| Critere | Pipeline traditionnel | LTX-2 conjoint |
|---------|----------------------|----------------|
| Etapes | 4+ | 1 |
| Synchronisation | Manuelle, imparfaite | Native, parfaite |
| Coherence audio-video | Faible (assemblee) | Elevee (apprise) |
| Cout compute | Somme des modeles | Un seul modele 22B |
| Flexibilite controle | Elevee (chaque etape) | Restreinte (un seul modele) |

Le trade-off : LTX-2 sacrifie la flexibilite (un seul modele, moins de controle granulaire par etape) mais gagne la coherence native. Pour des usages ou la synchro audio-video est critique (scenettes animees, paysages sonores), c'est un gain qualitatif decisif.


### Exercice 1 : Estimateur de VRAM pour LTX-2 avec quantization

**Objectif** : Implementer une fonction qui estime la consommation VRAM de LTX-2 (22B) en fonction de la quantization et des parametres de generation, et determine si une configuration est realisable sur le GPU disponible.

Le modele 22B fait ~44 GB en FP16. La quantization reduit drastiquement cet encombrement : INT8 ~ 22 GB, INT4/NF4 ~ 11 GB. Mais il faut aussi compter le text encoder Gemma 3 12B et les activations de generation.

**Indices** :
- # Etape 1 : Calculer le cout du modele selon la quantization. FP16 = 44 GB, INT8 = 22 GB, INT4/NF4 = 11 GB (regle : bits/16 * 44)
- # Etape 2 : Ajouter le cout du text encoder Gemma 3 12B (~6 GB en QAT quantized, ~24 GB en FP16)
- # Etape 3 : Ajouter le cout d'activation (~0.3 GB par 512x768x97 frames en INT4). Si total > VRAM disponible, proposer une config reduite (baisser frames puis resolution)
- # Indice : Le VAE slicing et le CPU offload reduisent encore la VRAM mais ralentissent la generation


In [8]:
def estimate_ltx2_vram(width: int, height: int, num_frames: int,
                       quantization: str = "int4_nf4",
                       gemma_quantized: bool = True) -> dict:
    '''
    Estime la VRAM requise pour LTX-2 avec une configuration donnee.

    Args:
        width: Largeur video
        height: Hauteur video
        num_frames: Nombre de frames
        quantization: "fp16", "int8", ou "int4_nf4"
        gemma_quantized: True si Gemma 3 est en version QAT quantized

    Returns:
        Dict avec : vram_estimee_gb, vram_modele_gb, vram_gemma_gb,
                    vram_activations_gb, faisable (bool), config_alternative (si non faisable)
    '''
    # Etape 1 : Cout du modele 22B selon la quantization
    # TODO etudiant : 44 GB en FP16, 22 en INT8, 11 en INT4/NF4
    pass

    # Etape 2 : Cout du text encoder Gemma 3 12B
    # TODO etudiant : ~6 GB si QAT quantized, ~24 GB sinon
    pass

    # Etape 3 : Cout d'activation + verification de faisabilite
    # TODO etudiant : ~0.3 GB par 512x768x97 frames en INT4, echelle avec width*height*num_frames
    pass

    return None  # TODO etudiant : retourner le dictionnaire d'estimation

# Test avec la configuration par defaut du notebook
print("Exercice a completer")


Exercice a completer


### Exercice 2 : Planificateur de prompts audiovisuels

**Objectif** : Creer une fonction qui compose un prompt LTX-2 exploitant la capacite conjointe audio+video du modele. Un bon prompt LTX-2 decrit a la fois la scene visuelle ET le son attendu, car le modele genere les deux ensemble.

Contrairement a un prompt video-only (ou le son est ajoute apres), un prompt LTX-2 doit guider la **coherence audio-visuelle** (un eclat de rire = un visage qui rit, un orage = des eclairs et du tonnerre synchrone).

**Indices** :
- # Etape 1 : Definir un dictionnaire `AV_CUES` associant des scenes a des paires (visuel, audio) coherentes (tempete -> eclairs + tonnerre ; foret -> feuilles + chants d'oiseaux ; rue -> voitures + klaxons)
- # Etape 2 : Detecter la scene dominante a partir de mots-cles dans la description visuelle
- # Etape 3 : Combiner la description visuelle + les indices audio coherents + des mots de qualite (cinematic, natural sound, synchronized) en un prompt LTX-2 unique
- # Indice : LTX-2 repond bien aux descriptions audio explicites ("natural ambient sound", "clear dialogue", "dynamic foley")


In [9]:
def build_av_prompt(visual_description: str, audio_description: str = "",
                   mood: str = "cinematic") -> str:
    '''
    Compose un prompt LTX-2 exploitant la generation audiovisuelle conjointe.

    Args:
        visual_description: Description de la scene visuelle
        audio_description: Description du son souhaite (vide = deduction auto)
        mood: Ambiance ("cinematic", "documentary", "dramatic", "peaceful")

    Returns:
        Prompt LTX-2 combine visuel + audio
    '''
    # Etape 1 : Definir les paires audio-visuelles coherentes par type de scene
    AV_CUES = {}  # TODO etudiant : tempete, foret, rue, plage, etc.

    # Etape 2 : Detecter la scene dominante
    # TODO etudiant : analyser visual_description pour trouver le type
    pass

    # Etape 3 : Combiner visuel + audio + mood
    # TODO etudiant : si audio_description vide, utiliser AV_CUES de la scene detectee
    pass

    return None  # TODO etudiant : retourner le prompt combine

# Test avec differentes scenes
print("Exercice a completer")


Exercice a completer


### Exercice 3 : Analyseur de synchronisation audio-video

**Objectif** : Implementer une fonction qui mesure la qualite de synchronisation entre la piste audio et la sequence video d'un MP4 genere par LTX-2. La synchronisation native est LE feature de LTX-2 ; cet exercice construit une metrique pour la quantifier.

L'idee : comparer l'**enveloppe d'energie audio** (ou les onsets sonores apparaissent) a l'**energie de mouvement** des frames (ou l'image change le plus). Une bonne synchro = les pics d'audio coïncident avec les pics de mouvement (un coup = un son au meme instant).

**Indices** :
- # Etape 1 : Extraire la piste audio du MP4 avec `av` (PyAV) et calculer l'enveloppe d'energie par fenetre (RMS ou amplitude absolue moyenne par frame audio)
- # Etape 2 : Calculer l'energie de mouvement entre frames video consecutives : `np.mean(np.abs(f_t - f_{t-1}))` redimensionne a la meme cadence que l'audio
- # Etape 3 : Calculer la cross-correlation decalee entre les deux enveloppes ; le lag au pic de correlation = decalage audio-video (ideal = 0). Un decalage < 40 ms est imperceptible
- # Indice : Utilisez `np.correlate(audio_env, motion_env, mode='full')` puis trouvez l'argmax


In [10]:
def analyze_av_sync(video_path: str) -> dict:
    '''
    Analyse la synchronisation audio-video d'un fichier MP4.

    Args:
        video_path: Chemin vers le fichier MP4 (avec piste audio)

    Returns:
        Dict avec : lag_ms, correlation_peak, sync_quality (0-100),
                    audio_envelope, motion_envelope, classification
    '''
    # Etape 1 : Extraire l'audio et calculer son enveloppe d'energie
    # TODO etudiant : utiliser av (PyAV) pour demuxer la piste audio, calculer RMS par fenetre
    pass

    # Etape 2 : Calculer l'energie de mouvement des frames video
    # TODO etudiant : iterer les frames, diff absolue moyenne consecutive
    pass

    # Etape 3 : Cross-correlation decalee et classification de la synchro
    # TODO etudiant : np.correlate(mode='full'), argmax -> lag, seuils <40ms imperceptible
    pass

    return None  # TODO etudiant : retourner le dictionnaire d'analyse

# Test (necessite un MP4 genere par LTX-2)
print("Exercice a completer")


Exercice a completer


## Section 3 : Comparaison LTX-2 vs LTX-Video 0.9

LTX-Video 0.9 (notebook 02-2) et LTX-2 partagent la lignee Lightricks mais different radicalement. Le tableau suivant resume les compromis pour choisir le bon modele selon le cas d'usage.

| Critere | LTX-Video 0.9 (02-2) | LTX-2 (02-5) |
|---------|----------------------|--------------|
| **Audio** | Non | Oui, conjoint |
| **Parametres** | ~2B | 22B |
| **VRAM** | ~8 GB (FP16) | ~16-24 GB (INT4/NF4) |
| **Vitesse** | Rapide (2-5x LTX-2) | Lent (22B) |
| **API** | diffusers standard | ltx-pipelines custom |
| **Qualite video** | Correcte | SOTA |
| **Licence** | Ouverte | LTX-2 Community (non-OSI) |
| **Cas d'usage** | Preview rapide, video muette | Production, video + audio synchronise |

### Quand utiliser LTX-2

| Scenario | Recommandation | Raison |
|----------|----------------|--------|
| Video muette rapide | LTX-Video 0.9 (02-2) | 5x plus rapide, VRAM reduite |
| Video + audio synchronise | LTX-2 (02-5) | Seul modele de la serie avec audio natif |
| Prototypage de prompt | LTX-Video 0.9 | Iterations rapides |
| Production finale lip-sync | LTX-2 | Synchro native, pas de post-traitement |
| GPU 8-12 GB | LTX-Video 0.9 | LTX-2 ne tient pas |
| Scene avec son diegetique (vagues, orage) | LTX-2 | Audio coherent appris |


In [11]:
# Statistiques de session et prochaines etapes
print("\n--- STATISTIQUES DE SESSION ---")
print("=" * 40)

print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}")
print(f"Modele : {model_repo} ({model_file})")
print(f"Device : {device}")
print(f"Quantization : {quantization}")
print(f"Parametres : {num_frames} frames, {num_inference_steps} steps, CFG={guidance_scale}")
print(f"Resolution : {width}x{height} | Audio : {audio}")

if device == "cuda" and cuda_ok:
    try:
        vram_peak = torch.cuda.max_memory_allocated(0) / 1024**3
        print(f"VRAM pic session : {vram_peak:.1f} GB")
    except Exception:
        print("VRAM pic session : n/a (init CUDA bloque)")

if save_results and OUTPUT_DIR.exists():
    generated_files = list(OUTPUT_DIR.glob('*'))
    print(f"\nFichiers generes ({len(generated_files)}) :")
    for f in sorted(generated_files):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name} ({size_kb:.1f} KB)")

print(f"\n--- PROCHAINES ETAPES ---")
print(f"1. Valider Gemma 3 (acceptation licence HF + token approuve)")
print(f"2. Re-executer ce notebook avec run_generation=True pour generation GPU reelle")
print(f"3. Module 03-1 : benchmark comparatif LTX-2 vs LTX-Video 0.9 vs HunyuanVideo")
print(f"4. Combiner LTX-2 audio-video avec le notebook Audio pour post-traitement foley")

print(f"\nNotebook 02-5 LTX-2 Audiovisual termine - {datetime.now().strftime('%H:%M:%S')}")



--- STATISTIQUES DE SESSION ---
Date : 2026-06-15 16:09:01
Mode : interactive
Modele : Lightricks/LTX-2.3 (ltx-2.3-22b-dev.safetensors)
Device : cuda
Quantization : int4_nf4
Parametres : 97 frames, 30 steps, CFG=3.0
Resolution : 768x512 | Audio : True
VRAM pic session : 0.0 GB

Fichiers generes (0) :

--- PROCHAINES ETAPES ---
1. Valider Gemma 3 (acceptation licence HF + token approuve)
2. Re-executer ce notebook avec run_generation=True pour generation GPU reelle
3. Module 03-1 : benchmark comparatif LTX-2 vs LTX-Video 0.9 vs HunyuanVideo
4. Combiner LTX-2 audio-video avec le notebook Audio pour post-traitement foley

Notebook 02-5 LTX-2 Audiovisual termine - 16:09:01


## Conclusion

Ce notebook a introduit **LTX-2**, le premier modele fondationnel audio-video base sur DiT. Les points cles :

- LTX-2 genere **video et audio synchronises en une seule passe** (nouveaute vs toute la serie Video)
- Le modele 22B necessite une **quantization INT4/NF4** pour tenir sur un GPU 24 GB
- L'API est **ltx-pipelines** (custom Lightricks), differente de l'API diffusers des autres notebooks
- La licence **LTX-2 Community** (non-OSI) impose un seuil commercial ; verifier avant usage production

### Etat d'execution

> **Notebook en attente d'execution GPU reelle.** Le checkpoint LTX-2.3 (44 GB) est telecharge. Le text encoder **Gemma 3 12B est gated sur HuggingFace** : le user doit accepter la licence sur la page du modele puis propager l'approbation au token avant que la generation puisse s'executer. Les cellules de code sont completes et correctes (API verifiee contre le repo Lightricks/LTX-2) ; une fois Gemma debloque, re-executer avec `run_generation=True` produira les outputs reels (regle C.2).

**Pour aller plus loin** : comparer LTX-2 avec le pipeline traditionnel (video-only + TTS separe) du module 03-Orchestration pour quantifier le gain de synchro native.

---

**Navigation :** [<< 02-4 SVD](02-4-SVD-Image-to-Video.ipynb) | [Index](../README.md) | [Comparaison multi-modeles >>](../03-Orchestration/03-1-Multi-Model-Video-Comparison.ipynb)
